##**Projeto:** Merca Data Platform

##**Squad:** 2 | Camada Gold
### Objetivo
Construir a **tabela fato de vendas** do Squad 2, enriquecendo os itens de pedido validados com dados de pedidos (via JOIN) e métricas calculadas de receita e desconto, disponibilizando o resultado para o SQL Server.
### Origem e Destinos
| Item | Valor |
| **Origem principal** | `squad2/silver/ecommerce_itens_pedido` (Delta Lake) |
| **Origem lookup** | `squad2/silver/ecommerce_pedidos` (Delta Lake) |
| **Destino Lake** | `squad2/gold/ecommerce_itens_pedido` (Delta Lake — append) |
| **Destino SQL** | `squad2.gold_ecommerce_itens_pedido` (SQL Server — append) |
| **Controle** | `gold/control/ecommerce_itens_pedido.json` |
### Regras de Negócio Aplicadas
| # | Regra | Descrição | Saída |
| 6 | SKUs únicos vendidos na última hora | Contagem de produtos distintos no período | Métrica de monitoramento |
| 7 | Top 5 SKUs em 30 minutos | Ranking dos mais vendidos no período | Métrica de monitoramento |
| 8 | Receita Bruta vs Líquida | Alerta se desconto > 25% da receita bruta | Log de alerta |
| 9 | Desconto impossível | `desconto_aplicado > preco_unitario` | Campo `alerta_desconto_impossivel` |
| 10 | Média de itens por pedido | Alerta se média < 3 itens/pedido | Log de alerta |
### Campos Calculados na Gold
| Campo | Lógica | Tipo |
| `receita_bruta_calculada` | `quantidade × preco_unitario` | float |
| `receita_liquida_calculada` | `receita_bruta_calculada − desconto_aplicado` | float |
| `alerta_desconto_impossivel` | `desconto_aplicado > preco_unitario` | `'SIM'`/`'NAO'` |
| `gold_processed_at` | Timestamp de processamento na Gold | datetime |
| `data_pedido` | Herdado via JOIN com `ecommerce_pedidos` | date |
| `id_cliente` | Herdado via JOIN com `ecommerce_pedidos` | string |
| `status` | Herdado via JOIN com `ecommerce_pedidos` | string |
### Dependências
| Notebook | Motivo |
| `feat_squad2_99_helpers` | `get_storage_options`, `get_squad2_client`, `SQL_OPTIONS` |
| `feat_squad2_silver_itens_pedido` | Dados validados de itens |
| Silver `ecommerce_pedidos` | Lookup de dados do pedido pai |


In [0]:
%run ../utils/feat_squad2_99_helpers

In [0]:
from deltalake import DeltaTable, write_deltalake
import pandas as pd
from datetime import datetime
import json

TABELA_ITENS = "ecommerce_itens_pedido"
TABELA_PEDIDOS = "ecommerce_pedidos"
TABELA_SQL = f"gold_{TABELA_ITENS}"  

path_silver_itens   = f"abfss://{SQUAD2_CONTAINER}@{ADLS_STORAGE_ACCOUNT}.dfs.core.windows.net/silver/{TABELA_ITENS}"
path_silver_pedidos = f"abfss://{SQUAD2_CONTAINER}@{ADLS_STORAGE_ACCOUNT}.dfs.core.windows.net/silver/{TABELA_PEDIDOS}"
path_gold           = f"abfss://{SQUAD2_CONTAINER}@{ADLS_STORAGE_ACCOUNT}.dfs.core.windows.net/gold/{TABELA_ITENS}"
path_control        = f"gold/control/{TABELA_ITENS}.json"

- Construção da Tabela Fato e Gravação

**Pré-condições verificadas:**
- Silver de itens deve estar inicializada
- Silver de pedidos deve estar inicializada (necessária para o JOIN)
**JOIN com Pedidos:** enriquece cada item com `data_pedido`, `id_cliente` e `status` do pedido pai.
O JOIN é `left` para não descartar itens cujo pedido não seja encontrado.
 **Métricas calculadas por linha:**
- `receita_bruta_calculada = quantidade × preco_unitario`
- `receita_liquida_calculada = receita_bruta - desconto_aplicado`
- `alerta_desconto_impossivel = 'SIM'` quando desconto supera o preço unitário
**Sink duplo:** dados gravados simultaneamente no Delta Lake e no SQL Server.

In [0]:
try:
    if not DeltaTable.is_deltatable(path_silver_itens, storage_options=get_storage_options()):
        print(f" [Aviso Gold] A tabela Silver de Itens {TABELA_ITENS} ainda não foi inicializada.")
    elif not DeltaTable.is_deltatable(path_silver_pedidos, storage_options=get_storage_options()):
        print(f" [Aviso Gold] A tabela Silver de Pedidos {TABELA_PEDIDOS} não foi localizada.")
    else:
        dt_itens = DeltaTable(path_silver_itens, storage_options=get_storage_options())
        df_itens_pandas = dt_itens.to_pandas()
        
        squad2_client = get_squad2_client()
        file_client = squad2_client.get_file_client(path_control)
        
        processados = set()
        if file_client.exists():
            conteudo = file_client.download_file().readall().decode('utf-8')
            processados = set(json.loads(conteudo))
        
        # Comentamos a linha incremental e forçamos a leitura de toda a base Silver
        df_itens_novos = df_itens_pandas[~df_itens_pandas['bronze_source_file'].isin(processados)].copy()
        #df_itens_novos = df_itens_pandas.copy() -- Forçar para testes
        
        if df_itens_novos.empty:
            print(" Camada Gold de Itens de Pedido em dia! Nenhum registro novo para processar.")
        else:
            print(f"  Processando {len(df_itens_novos)} linhas para a Fato. Realizando lookup na base de Pedidos...")
            
            dt_pedidos = DeltaTable(path_silver_pedidos, storage_options=get_storage_options())
            df_pedidos_pandas = dt_pedidos.to_pandas()
            
            colunas_pedidos = ['id_pedido', 'data_pedido', 'id_cliente', 'status']
            colunas_existentes = [col for col in colunas_pedidos if col in df_pedidos_pandas.columns]
            df_pedidos_lookup = df_pedidos_pandas[colunas_existentes].drop_duplicates(subset=['id_pedido'])
            
            # Executa o JOIN da Tabela Fato
            df_gold_final = pd.merge(df_itens_novos, df_pedidos_lookup, on='id_pedido', how='left')
            
            # Regras calculadas por linha
            df_gold_final['receita_bruta_calculada'] = df_gold_final['quantidade'] * df_gold_final['preco_unitario']
            df_gold_final['receita_liquida_calculada'] = df_gold_final['receita_bruta_calculada'] - df_gold_final['desconto_aplicado']
            df_gold_final['alerta_desconto_impossivel'] = df_gold_final.apply(
                lambda row: 'SIM' if row['desconto_aplicado'] > row['preco_unitario'] else 'NAO', axis=1
            )
            
            # Coluna de Auditoria e Verificação de Atualização
            df_gold_final['gold_processed_at'] = datetime.now()
            
            for col in df_gold_final.columns:
                if pd.api.types.is_datetime64_any_dtype(df_gold_final[col]):
                    df_gold_final[col] = df_gold_final[col].dt.tz_localize(None)
            
            # SINK 1: Gravação Lakehouse (Pasta Gold)
            write_deltalake(path_gold, df_gold_final, mode="append", storage_options=get_storage_options())
            
            # -------------------------------------------------------------------------
            # SINK 2: INGESTÃO NO SQL SERVER COM CRIAÇÃO E ALINHAMENTO AUTOMÁTICO
            # -------------------------------------------------------------------------
            try:
                df_schema_sql = spark.read \
                    .format("sqlserver") \
                    .options(**SQL_OPTIONS) \
                    .option("dbtable", f"[squad2].[{TABELA_SQL}]") \
                    .load() \
                    .limit(0)
                
                spark_df_final = spark.createDataFrame(df_gold_final)
                for col_db in df_schema_sql.columns:
                    col_match = [c for c in spark_df_final.columns if c.lower() == col_db.lower()]
                    if col_match:
                        spark_df_final = spark_df_final.withColumnRenamed(col_match[0], col_db)
                spark_df_aligned = spark_df_final.select(*df_schema_sql.columns)
                print(f"  Tabela existente localizada. Alinhando colunas e fazendo Append...")
            except Exception:
                print(f"  Criando nova tabela de Fato diferenciada: [squad2].[{TABELA_SQL}]...")
                spark_df_aligned = spark.createDataFrame(df_gold_final)
            
            spark_df_aligned.write \
                .format("sqlserver") \
                .options(**SQL_OPTIONS) \
                .option("dbtable", f"[squad2].[{TABELA_SQL}]") \
                .mode("append") \
                .save()
            
            # Atualiza o controle incremental
            arquivos_atuais = set(df_itens_novos['bronze_source_file'].unique())
            todos_processados = list(processados.union(arquivos_atuais))
            file_client.upload_data(json.dumps(todos_processados), overwrite=True)
            
            print(f"\n SUCESSO! Itens enriquecidos salvos em squad2.{TABELA_SQL}!")

except Exception as e:
    print(f"  Erro no processamento: {str(e)}")
    raise